# THEREDA → GEMS3 Export

Exports the ThermoHub-schema graph produced by `import-THEREDA-json-data.ipynb` into GEMS3 backup
format (`IComp`/`DComp`/`ReacDC`/`Phase`/`SDref` JSON) via `ExportToGems3`. Run this after the
import notebook has populated the target database.

Equivalent standalone script: `export-gems3.py` (same three steps, runnable headlessly via
`python export-gems3.py` — useful for testing a fix without touching a live Jupyter kernel, since a
running kernel can't hot-reload a rebuilt `thermomatch` extension).

In [1]:
import thermomatch as match
match.ThermoImpexGenerator.create_local_database_if_not_existent = True

Same `thermomatch` import as the import notebook. No `create_local_database_if_not_existent` flag
here — export assumes the target database already exists and was populated by the import
notebook.

In [2]:
# Set path to `schemas` and `lua` not from the configuration file
match.ThermoImpexGenerator.setResourcesDirectory("../../Resources")

# THEREDA-specific redox/formula/symbol-shortening/SDref-author rules - used at both import time
# (formula charge-fixing) and export time (symbol shortening, SDref author keys); process-wide,
# so load it before both runImport() and ExportAllJson() below.
match.loadGems3ExportRulesFile("scripts-out/gems3-export-rules.json")

[2026-08-05 14:59:44.328] [jsonio17] [info] Home directory is /home/dmiron
[thermomatch] [info] jsonio17::ioSettings() is reading settings from "../../Resources/ThermoMatch-config.json" (resolved to "/home/dmiron/git/hub/thermoimpex-jupyter/databases/THEREDA/../../Resources/ThermoMatch-config.json", cwd "/home/dmiron/git/hub/thermoimpex-jupyter/databases/THEREDA")
[thermomatch] [info] jsonio17::ioSettings() common.ResourcesDirectory = "../../Resources"
[thermomatch] [info] Loaded Gems3Export rules from "scripts-out/gems3-export-rules.json" (RedoxElements=29 FormulaOverrides=1 SymbolShortenRules=15 SDrefAuthorKeyOverrides=3 ModeCodeRules=6)


`loadGems3ExportRulesFile(...)` loads THEREDA's redox-element list, formula overrides, GEMS3
symbol-shortening regex rules, SDref author-key overrides, and `TPcMod`/`REcMod` mode-code override
rules — all as **process-wide** state (not tied to any one `ExportToGems3` instance), consumed by
`checkAndFixChargeFormula`, `shortenDCSymbol`, `ExportSDrefJson`, and `applyModeCodeRules`
respectively.

**Must be loaded before `ExportToGems3(...)` is constructed below** (and before any export calls
run) — there is no hardcoded C++ default any more, so skipping this call means zero redox-capable
elements, formula overrides, or symbol-shortening rules get applied during export.

In [ ]:
#Export to GEMS3k format
gem_export = match.ExportToGems3("http://localhost:8529", "root", "", "ORD_THEREDA_2026-01", "thermodatasets/THEREDA2026;1:TDS_LMA;0");

[jsonarango] [info] You are connected to: arango 3.9.12
[jsonio17] [info] VertexSubstance loading collection: 8196, loading query: 121915
[jsonio17] [info] VertexPhase loading collection: 1163, loading query: 60062
[jsonio17] [info] VertexInteraction loading collection: 1855, loading query: 193750
[jsonio17] [info] VertexDataSource loading collection: 2143, loading query: 23897
[thermomatch] [warning] Substance "(KCl)3(PbCl2)3H2O(s)": symbol could not be shortened to fit the GEMS3 16-character DC symbol limit (best effort: "(KCl)3(PbCl2)3w(s)", 18 characters); it will be truncated and disambiguated with a numeric suffix. Consider adding a "tag" for this substance, or a rule to the loaded Gems3Export rules file's "SymbolShortenRules".
[thermomatch] [warning] Substance "(MgCl2)3(PbCl2)19H2O(s)": symbol could not be shortened to fit the GEMS3 16-character DC symbol limit (best effort: "(MgCl2)3PbCl2w19(s)", 19 characters); it will be truncated and disambiguated with a numeric suffix. Cons

The 5-arg `ExportToGems3` constructor's last parameter is overloaded: a bare `idThermoDataSet`
string (as used here, `"thermodatasets/THEREDA2026;1:TDS_LMA;0"`) selects everything linked to that
thermodataset via `select_linked()`. Passing the same string wrapped in a Python list instead
silently binds to a *different* overload (`reaction_keys: vector<string>`) and produces an empty
selection with no error — keep this a plain string, not `[idThermoDataSet]`.

In [4]:
gem_export.ExportAllJson("data-out");

[jsonio17] [info] VertexElement loading collection: 49, loading query: 5805
[jsonio17] [info] VertexSubstance loading collection: 22, loading query: 89799
[jsonio17] [info] VertexReaction loading collection: 1478, loading query: 70570
[jsonio17] [info] VertexPhase loading collection: 81, loading query: 74318
[thermomatch] [warning] Phase "(CdCl2)3(Cd(OH2))5(cr)": "symbol" could not be shortened to fit the GEMS3 16-character phase name limit (best effort: "(CdCl2)3(CdOH2)5(cr)", 20 characters); it will be truncated and disambiguated with a numeric suffix. Consider adding a rule to the loaded Gems3Export rules file's "SymbolShortenRules".
[thermomatch] [warning] Phase "(CdCl2)3(Cd(OH2))5(cr)": "name" could not be shortened to fit the GEMS3 16-character phase name limit (best effort: "(CdCl2)3(CdOH2)5(cr)", 20 characters); it will be truncated and disambiguated with a numeric suffix. Consider adding a rule to the loaded Gems3Export rules file's "SymbolShortenRules".
[thermomatch] [warning

Writes `IComp`/`DComp`/`ReacDC`/`SDref` JSON into `data-out`, plus phases split across two files:
`Phase.<name>.json` (solid-solution/mixing-model phases, plus any pure phase whose substance isn't
part of a solid solution) and `Phase.<name>.pure.json` (every `mixmod==0` pure phase, including
ones already covered by the mixed file). If `log.module_level.thermomatch` is set to `"debug"` in
`ThermoMatch-config.json`, this call also dumps the full per-reaction JSON to the log for every
reaction — harmless for a plain script, but can overwhelm Jupyter's output rendering.

Known post-export caveat noted below: GEMS3 flips the sign of the second Redlich-Kister parameter
relative to how it's stored here, so exported `RK_...` phase files may need that sign corrected by
hand before use in GEMS3.

In [5]:
# in GEMS second RK parameters changes sign RK_(SO4)Ca2Al0.6	1	0	1.67	-0.946 --> RK_(SO4)Ca2Al0.6	1	0	1.67	0.946

## Also build GEM-Selektor `.pdb`/`.ndx` files (via `json2db`)

`ExportAllJson(...)` above only writes the intermediate `*.backup.json` files. This step converts
each of them into a real GEM-Selektor `<keyword>.<tag>.pdb`/`.ndx` pair in `data-out/gems-auto`,
using GEMSGUI's headless `json2db` CLI tool (see `gemsgui_import.py` at the repo root, and
GEMSGUI's own `CLAUDE.md`, "benchcomp/json2db" section, for how that tool works). Requires
`json2db` to be installed (`bash conda-install-dependencies.sh`, run once per environment).

The two `Phase.backup*.json` files both map to the `phase` keyword but overlap on some keys (per
the note above, `Phase.backup.pure.json` re-includes every pure phase also present in
`Phase.backup.json`) — writing them into the same `.pdb` would silently let the pure-phase version
win on any overlapping key, so they get distinct tags (`...` and `....pure`) instead.

In [3]:
import sys
sys.path.append("../..")
from gemsgui_import import json_to_db

THEREDA_TAG = "THEREDA.ver2026-01-r17"

for json_name, keyword, tag in [
    ("IComp.backup.json", "icomp", THEREDA_TAG),
    ("DComp.backup.json", "dcomp", THEREDA_TAG),
    ("Phase.backup.json", "phase", THEREDA_TAG),
    ("Phase.backup.pure.json", "phase", THEREDA_TAG + ".pure"),
    ("ReacDC.backup.json", "reacdc", THEREDA_TAG),
    ("SDref.backup.json", "sdref", THEREDA_TAG),
]:
    json_to_db(f"data-out/{json_name}", keyword=keyword, output_dir="data-out/gems-auto", tag=tag)
print("done")

done
